<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Alains_Curve_Animation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Alain's Curve Visualization Project

## Overview
This notebook contains a high-fidelity mathematical animation of Alain's Curve using the Manim library. The implementation focuses on a parameter sweep that demonstrates how the curve's topology transitions as the parameter 'a' changes. The visualization is styled with an 'Aged Paper and Fountain Pen' aesthetic, designed for a 9:16 vertical aspect ratio.

## Mathematical Background
Alain's Curve is a type of quartic curve defined by the implicit equation:

$$(x^2 - y^2)^2 = a^2x^2 - b^2y^2$$

### Key Characteristics
1. **Quartic Nature**: As a degree 4 polynomial equation, it exhibits complex behavior including loops and asymptotic branches.
2. **Parameter Dynamics**:
   - The parameter **a** primarily influences the width and existence of the inner loops and the spread of the horizontal branches.
   - The parameter **b** affects the vertical scaling and the steepness of the branches as they approach their limits.
3. **Asymptotes**: The curve is guided by the lines $y = \pm x$, which act as asymptotic boundaries for the branches as the coordinates grow larger.
4. **Topology Transitions**: Small values of 'a' relative to 'b' typically result in more localized, loop-like structures, while increasing 'a' causes the curve to expand into wider, sweeping branches.

## Implementation Details
- **Rendering**: Uses Manim's Community Edition with custom configuration for vertical mobile formats (720x1280).
- **Aesthetics**: Implements a procedural paper texture generation and an 'ink bleed' effect for all vector objects to simulate a hand-drawn calligraphic look.
- **Animation**: Employs a ValueTracker to perform a live sweep of the parameter 'a', leaving persistent 'stamps' at specific intervals to show the history of the transformation.

In [2]:
!apt-get update -qq
!apt-get install -y -qq libcairo2-dev libpango1.0-dev ffmpeg dvisvgm texlive-latex-extra texlive-fonts-extra
!pip install -q manim


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package fonts-droid-fallback.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../000-fonts-droid-fallback_1%3a6.0.1r16-1.1build1_all.deb ...
Unpacking fonts-droid-fallback (1:6.0.1r16-1.1build1) ...
Selecting previously unselected package fonts-lato.
Preparing to unpack .../001-fonts-lato_2.0-2.1_all.deb ...
Unpacking fonts-lato (2.0-2.1) ...
Selecting previously unselected package poppler-data.
Preparing to unpack .../002-poppler-data_0.4.11-1_all.deb ...
Unpacking poppler-data (0.4.11-1) ...
Selecting previously unselected package tex-common.
Preparing to unpack .../003-tex-common_6.17_all.deb ...
Unpacking tex-common (6.17) ...
Selecting previously unse

In [1]:
import manim
from manim.utils.ipython_magic import ManimMagic

try:
    # Manually register the %%manim magic command into the IPython shell
    get_ipython().register_magics(ManimMagic)
    print(f"Manim {manim.__version__} loaded and magic commands registered successfully.")
except Exception as e:
    print(f"Error loading Manim magic: {e}")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Manim 0.20.1 loaded and magic commands registered successfully.


In [6]:
# Cell 2 — render
%%manim -qm --resolution 720,1280 AlainsCurveShortsFixed
"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
Concept: Alain's Curve — parameter sweep and branch transition

Refactored for 9:16 robust layout, "Aged Paper & Leaky Fountain Pen" aesthetics,
persistent curve stacking, and asymptotic bug fixes.
"""

from manim import *
import numpy as np
import random

# Shorts framing enforced in config
config.pixel_width = 720
config.pixel_height = 1280
config.frame_width = 9
config.frame_height = 16
config.background_color = "#E5D8C1"  # Base Aged Paper Color
config.media_dir = "manim_media"

A_MIN = 0.4
A_MAX = 1.3
B_FIXED = 1.2

# Color Palette: Paper and Fountain Pen Ink
INK = "#1A2530"           # Deep, wet dark blue/black ink
FADED_INK = "#5C6B7A"     # Faded text and pencil grids
ACCENT_INK = "#821E14"    # Red editing ink
PAPER_FILL = "#EADFCB"    # Slightly lighter patch for UI cards
PEN_FONT = "TeX Gyre Chorus"  # Calligraphic font

# Expanded 7-step sweep
A_SWEEP = [0.4, 0.5, 0.6, 0.9, 1.0, 1.1, 1.3]
A_COLORS = [
    "#264653", # Faded Teal
    "#216869", # Viridian
    "#2A4B7C", # Navy
    "#4A192C", # Deep Purple
    "#7A1C4B", # Magenta/Plum
    "#9D0208", # Crimson
    "#5C0000"  # Blood Red
]

def apply_ink_bleed(vgroup, color_hex, base_width=4.0):
    """Simulates fountain pen ink bleeding into paper fibers."""
    bleed_group = VGroup()
    # 1. Outer bleed (wide, very transparent)
    bleed1 = vgroup.copy().set_stroke(color=color_hex, width=base_width * 4.0, opacity=0.06)
    # 2. Inner bleed (medium width, translucent)
    bleed2 = vgroup.copy().set_stroke(color=color_hex, width=base_width * 2.0, opacity=0.18)
    # 3. Core ink (tight, dark)
    core = vgroup.copy().set_stroke(color=color_hex, width=base_width, opacity=0.9)

    bleed_group.add(bleed1, bleed2, core)
    return bleed_group

def interpolate_a_color(a: float):
    t = np.clip((a - A_MIN) / (A_MAX - A_MIN), 0, 1)
    x = t * (len(A_COLORS) - 1)
    i = int(np.floor(x))
    if i >= len(A_COLORS) - 1:
        return A_COLORS[-1]

    c1 = ManimColor(A_COLORS[i])
    c2 = ManimColor(A_COLORS[i + 1])
    return c1.interpolate(c2, x - i)

def smooth_curve_segments(a: float, b: float, samples_per_octant: int = 180, eps: float = 1e-4):
    segments = []
    for k in range(8):
        t0 = k * np.pi / 4 + eps
        t1 = (k + 1) * np.pi / 4 - eps
        if t1 <= t0:
            continue

        thetas = np.linspace(t0, t1, samples_per_octant)
        current = []

        for t in thetas:
            denom = np.cos(2 * t) ** 2
            num = a * a * np.cos(t) ** 2 - b * b * np.sin(t) ** 2

            if denom > 1e-9 and num >= 0:
                r = np.sqrt(num / denom)
                x = r * np.cos(t)
                y = r * np.sin(t)

                # BUG FIX: Constrain coordinates directly to the graph viewport bounds [-4.0, 4.0]
                # This clips calculation immediately when branches leave the visible grid layout area
                if abs(x) > 4.0 or abs(y) > 4.0:
                    if len(current) >= 3:
                        segments.append(np.array(current))
                    current = []
                else:
                    current.append((x, y))
            else:
                if len(current) >= 3:
                    segments.append(np.array(current))
                current = []

        if len(current) >= 3:
            segments.append(np.array(current))

    return segments

def make_alain_curve(axes, a_value: float, b_value: float):
    group = VGroup()
    for seg in smooth_curve_segments(a_value, b_value):
        pts = [axes.c2p(float(x), float(y)) for x, y in seg]
        if len(pts) < 3:
            continue
        mob = VMobject()
        mob.set_points_smoothly(pts)
        mob.set_fill(opacity=0)
        group.add(mob)
    return group

class AlainsCurveShortsFixed(Scene):
    def generate_paper_texture(self):
        """Generates procedural irregular noise and water stains for the background texture."""
        bg = FullScreenRectangle(fill_color=config.background_color, fill_opacity=1, stroke_width=0)
        mottle = VGroup()

        # Scatter semi-transparent irregular blobs to look like realistic fluid/ink stains
        for _ in range(200):
            x = random.uniform(-config.frame_width/2, config.frame_width/2)
            y = random.uniform(-config.frame_height/2, config.frame_height/2)
            base_r = random.uniform(0.1, 0.75)
            alpha = random.uniform(0.01, 0.04)
            stain_color = random.choice(["#7A6A53", "#C4B299", "#524436", "#FFFFFF"])

            # Construct a fully irregular, organic blob polygon with smooth curves
            num_vertices = random.randint(6, 11)
            blob_points = []
            for i in range(num_vertices):
                angle = i * (2 * np.pi / num_vertices) + random.uniform(-0.15, 0.15)
                jittered_radius = base_r * random.uniform(0.4, 1.6)
                blob_points.append([
                    x + jittered_radius * np.cos(angle),
                    y + jittered_radius * np.sin(angle),
                    0
                ])

            stain = VMobject()
            stain.set_points_smoothly([*blob_points, blob_points[0]])
            stain.set_fill(color=stain_color, opacity=alpha)
            stain.set_stroke(width=0)
            mottle.add(stain)

        return VGroup(bg, mottle).set_z_index(-10)

    def construct(self):
        # 1. Background Texture
        paper_texture = self.generate_paper_texture()
        self.add(paper_texture)

        watermark = Text(
            "© Mugambi Ndwiga / @craftsandengineering",
            font_size=16,
            font=PEN_FONT,
            color=FADED_INK
        ).set_opacity(0.60).to_corner(DOWN + RIGHT, buff=0.18).set_z_index(5)
        self.add(watermark)

        # 2. Top Banner
        title = Text("Alain's Curve", font_size=50, font=PEN_FONT, color=INK)
        equation = MathTex(
            r"(x^2-y^2)^2=a^2x^2-b^2y^2",
            font_size=42,
            color=INK
        )
        subtitle = Text(
            "A parameter sweep through a quartic curve",
            font_size=20,
            font=PEN_FONT,
            color=FADED_INK
        )
        top_group = VGroup(title, equation, subtitle).arrange(DOWN, buff=0.15)

        # Adding a slight ink bleed to the card border itself
        top_card_border = SurroundingRectangle(
            top_group,
            corner_radius=0.18,
            buff=0.35,
            stroke_width=2.5,
            fill_color=PAPER_FILL,
            fill_opacity=0.85
        )
        top_card_inky = apply_ink_bleed(top_card_border, INK, base_width=1.5)
        top_ui = VGroup(top_card_inky, top_group).to_edge(UP, buff=0.6).set_z_index(5)

        # 3. Dynamic Trackers
        a_tracker = ValueTracker(A_MIN)

        current_a_label = Text("Current a", font_size=22, font=PEN_FONT, color=FADED_INK)
        current_a_value = always_redraw(
            lambda: MathTex(
                rf"a = {a_tracker.get_value():.1f}",
                font_size=32,
                color=interpolate_a_color(a_tracker.get_value())
            )
        )
        current_a_group = VGroup(current_a_label, current_a_value).arrange(DOWN, buff=0.08, aligned_edge=LEFT)

        fixed_b_label = Text("Fixed parameter", font_size=20, font=PEN_FONT, color=FADED_INK)
        fixed_b_value = MathTex(r"b = 1.2", font_size=28, color=INK)

        sweep_label = Text("Sweep range", font_size=20, font=PEN_FONT, color=FADED_INK)
        sweep_value = MathTex(r"a: 0.4 \rightarrow 1.3", font_size=28, color=INK)

        legend_rows = VGroup(
            *[
                MathTex(rf"a = {a:.1f}", font_size=24, color=c)
                for a, c in zip(A_SWEEP, A_COLORS)
            ]
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.1)

        # 4. Left Info Card
        info_text = VGroup(
            current_a_group,
            fixed_b_label,
            fixed_b_value,
            sweep_label,
            sweep_value,
            Text("Color tracks a", font_size=20, font=PEN_FONT, color=FADED_INK),
            legend_rows
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.25)

        info_card_border = SurroundingRectangle(
            info_text,
            corner_radius=0.18,
            buff=0.3,
            stroke_width=2.5,
            fill_color=PAPER_FILL,
            fill_opacity=0.85
        )
        info_card_inky = apply_ink_bleed(info_card_border, FADED_INK, base_width=1.5)
        left_ui = VGroup(info_card_inky, info_text).to_edge(LEFT, buff=0.3).shift(DOWN * 1.5).set_z_index(5)

        # 5. Main Plot Area
        axes = Axes(
            x_range=[-4.0, 4.0, 1],
            y_range=[-4.0, 4.0, 1],
            x_length=5.6,
            y_length=5.6,
            axis_config={
                "include_ticks": True,
                "stroke_width": 2.0,
                "color": FADED_INK
            },
            tips=True,
        )
        axes.next_to(left_ui, RIGHT, buff=0.30).align_to(left_ui, UP).shift(UP * 0.2)
        axes_inky = apply_ink_bleed(axes, FADED_INK, base_width=1.5)

        x_label = MathTex("x", font_size=32, color=INK).next_to(axes.c2p(4.05, 0), RIGHT, buff=0.08)
        y_label = MathTex("y", font_size=32, color=INK).next_to(axes.c2p(0, 4.05), UP, buff=0.06)
        labels = VGroup(x_label, y_label).set_z_index(4)

        guide_1 = DashedLine(
            axes.c2p(-4.05, -4.05),
            axes.c2p(4.05, 4.05),
            dash_length=0.16,
            stroke_width=1.5,
        ).set_opacity(0.4)
        guide_1_inky = apply_ink_bleed(guide_1, FADED_INK, base_width=1.0)

        guide_2 = DashedLine(
            axes.c2p(-4.05, 4.05),
            axes.c2p(4.05, -4.05),
            dash_length=0.16,
            stroke_width=1.5,
        ).set_opacity(0.4)
        guide_2_inky = apply_ink_bleed(guide_2, FADED_INK, base_width=1.0)

        # The "Live" sweeping curve
        live_curve = always_redraw(
            lambda: apply_ink_bleed(
                make_alain_curve(axes, a_tracker.get_value(), B_FIXED),
                interpolate_a_color(a_tracker.get_value()),
                base_width=3.5
            ).set_z_index(3)
        )

        # 6. Bottom Note Card
        note_text = Text(
            "As a increases, the inner loop gives way to wider branches.",
            font_size=18,
            font=PEN_FONT,
            color=INK
        )
        note_card_border = SurroundingRectangle(
            note_text,
            corner_radius=0.16,
            buff=0.25,
            stroke_width=1.5,
            fill_color=PAPER_FILL,
            fill_opacity=0.85
        )
        note_card_inky = apply_ink_bleed(note_card_border, FADED_INK, base_width=1.0)
        note_ui = VGroup(note_card_inky, note_text).to_edge(DOWN, buff=0.8).set_z_index(5)

        # --- ANIMATION SEQUENCE ---

        self.play(FadeIn(top_ui, shift=UP * 0.15), run_time=0.8)
        self.play(FadeIn(left_ui, shift=LEFT * 0.15), run_time=0.75)
        self.play(
            FadeIn(axes_inky), FadeIn(labels),
            FadeIn(guide_1_inky), FadeIn(guide_2_inky),
            run_time=0.9
        )

        # Fade in the first persistent stamp and the live tracker
        first_stamp = apply_ink_bleed(make_alain_curve(axes, A_SWEEP[0], B_FIXED), A_COLORS[0], base_width=3.5).set_z_index(2)
        self.play(Create(first_stamp), run_time=0.9)
        self.add(live_curve)

        self.play(FadeIn(note_ui, shift=UP * 0.1), run_time=0.5)
        self.wait(0.2)

        # Sweep and drop persistent stamps
        for i in range(1, len(A_SWEEP)):
            target_val = A_SWEEP[i]
            target_color = A_COLORS[i]

            # Animate the slider to the next step
            self.play(
                a_tracker.animate.set_value(target_val),
                run_time=1.3,
                rate_func=smooth
            )

            # Create a permanent, stationary copy of the curve at this location
            stamp = apply_ink_bleed(
                make_alain_curve(axes, target_val, B_FIXED),
                target_color,
                base_width=3.5
            ).set_z_index(2)
            self.add(stamp) # Locks it onto the paper

        self.wait(0.4)

        # Expert Note Switch
        expert_text = Text(
            "Expert note: the curve is quartic, and the diagonals y = ±x are asymptotic guides.",
            font_size=15,
            font=PEN_FONT,
            color=ACCENT_INK
        )
        expert_card_border = SurroundingRectangle(
            expert_text,
            corner_radius=0.14,
            buff=0.25,
            stroke_width=1.8,
            fill_color=PAPER_FILL,
            fill_opacity=0.85
        )
        expert_card_inky = apply_ink_bleed(expert_card_border, ACCENT_INK, base_width=1.0)
        expert_ui = VGroup(expert_card_inky, expert_text).move_to(note_ui).set_z_index(5)

        self.play(FadeOut(note_ui, shift=DOWN * 0.1), run_time=0.35)
        self.play(FadeIn(expert_ui, shift=UP * 0.1), run_time=0.55)
        self.wait(1.5)

        # Clean Fade Out
        closing_bg = FullScreenRectangle(fill_color=config.background_color, fill_opacity=1).set_z_index(10)

        self.play(FadeIn(closing_bg, run_time=0.6))
        self.wait(0.5)

Manim Community v0.20.1

[06/14/26 07:32:13] INFO     Animation 0 : Partial movie file written in                   ]8;id=209109;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=757541;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#601\601]8;;\
                             '/content/manim_media/videos/content/1280p30/partial_movie_fi                         
                             les/AlainsCurveShortsFixed/647511667_520976821_1216778632.mp4                         
                             '                                                                                     

[06/14/26 07:32:19] INFO     Animation 1 : Partial movie file written in                   ]8;id=736518;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=365234;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#601\601]8;;\
                             '/content/manim_media/videos/content/1280p30/partial_movie_fi                         
                             les/AlainsCurveShortsFixed/2942066320_1086200690_1746326657.m                         
                             p4'                                                                                   

[06/14/26 07:32:24] INFO     Animation 2 : Partial movie file written in                   ]8;id=501932;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=953597;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#601\601]8;;\
                             '/content/manim_media/videos/content/1280p30/partial_movie_fi                         
                             les/AlainsCurveShortsFixed/2942066320_898049499_3532665257.mp                         
                             4'                                                                                    

[06/14/26 07:32:30] INFO     Animation 3 : Partial movie file written in                   ]8;id=556936;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=264602;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#601\601]8;;\
                             '/content/manim_media/videos/content/1280p30/partial_movie_fi                         
                             les/AlainsCurveShortsFixed/2942066320_3004874599_3374041973.m                         
                             p4'                                                                                   

...

[06/14/26 07:35:34] INFO     Animation 16 : Partial movie file written in                  ]8;id=550557;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=528278;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#601\601]8;;\
                             '/content/manim_media/videos/content/1280p30/partial_movie_fi                         
                             les/AlainsCurveShortsFixed/2942066320_3468209647_1727901866.m                         
                             p4'                                                                                   

[06/14/26 07:35:39] INFO     Animation 17 : Partial movie file written in                  ]8;id=849194;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=15309;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#601\601]8;;\
                             '/content/manim_media/videos/content/1280p30/partial_movie_fi                         
                             les/AlainsCurveShortsFixed/2942066320_2574473371_92981027.mp4                         
                             '                                                                                     

                    INFO     Combining to Movie file.                                      ]8;id=420013;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=131509;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#753\753]8;;\

                    INFO                                                                   ]8;id=195718;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=282211;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#904\904]8;;\
                             File ready at                                                                         
                             '/content/manim_media/videos/content/1280p30/AlainsCurveShort                         
                             sFixed.mp4'                                                                           
                                                                                                                   

                    INFO     Rendered AlainsCurveShortsFixed                                           ]8;id=837954;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene.py\scene.py]8;;\:]8;id=663376;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene.py#278\278]8;;\
                             Played 18 animations                                                                  